 Veri setinin yapısal analizi ve ön incelemesi

In [1]:
import pandas as pd

PATH = "birlesik_veri.csv"
df = pd.read_csv(PATH)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))

print("\nTarget dağılımı (vehicle_type):")
print(df["vehicle_type"].value_counts())

print("\nEksik değer sayıları:")
print(df.isna().sum())

df.head(3)

Shape: (4599270, 9)

Columns: ['hour', 'passenger_count', 'vehicle_type', 'route_code', 'stop_code', 'district', 'is_outlier', 'is_peak_hour', 'mean_passenger_by_route_hour']

Target dağılımı (vehicle_type):
vehicle_type
2    1533090
1    1533090
3    1533090
Name: count, dtype: int64

Eksik değer sayıları:
hour                                0
passenger_count                     0
vehicle_type                        0
route_code                          0
stop_code                           0
district                        48880
is_outlier                          0
is_peak_hour                        0
mean_passenger_by_route_hour        0
dtype: int64


,hour,passenger_count,vehicle_type,route_code,stop_code,district,is_outlier,is_peak_hour,mean_passenger_by_route_hour
0,0,2,2,KIRAZLI-BASAKSEHIR/METROKENT,RAYLI,BAGCILAR,0,0,1.532982
1,0,2,2,YENIKAPI - HAVALIMANI,RAYLI,BAHCELIEVLER,0,0,2.980226
2,0,1,2,BAHARIYE-OLIMPIYAT,RAYLI,BAKIRKOY,0,0,1.473945


Gradient Boosting modeli için veri hazırlama ve eğitim süreci


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier


target = "vehicle_type"
X = df.drop(columns=[target]).copy()
y = df[target].astype(int).copy()


X["district"] = X["district"].fillna("UNKNOWN")


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Split shapes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Kolonlar
num_cols = ["hour", "passenger_count", "is_outlier", "is_peak_hour", "mean_passenger_by_route_hour"]
cat_cols = ["route_code", "stop_code", "district"]


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

#  Model: HistGradientBoosting (advanced + early stopping)
model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_depth=8,
    max_leaf_nodes=64,
    min_samples_leaf=40,
    l2_regularization=1e-4,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", model)
])

# Train
pipe.fit(X_train, y_train)

#   (Test)
pred = pipe.predict(X_test)

print("\n=== TEST Classification Report ===")
print(classification_report(y_test, pred, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, pred))


Split shapes:
Train: (3219489, 8) Val: (689890, 8) Test: (689891, 8)

=== TEST Classification Report ===
              precision    recall  f1-score   support

           1     1.0000    1.0000    1.0000    229963
           2     1.0000    1.0000    1.0000    229964
           3     1.0000    1.0000    1.0000    229964

    accuracy                         1.0000    689891
   macro avg     1.0000    1.0000    1.0000    689891
weighted avg     1.0000    1.0000    1.0000    689891


=== Confusion Matrix ===
[[229963      0      0]
 [     0 229964      0]
 [     0      0 229964]]


Veri sızıntısı giderilerek Gradient Boosting modelinin eğitilmesi ve değerlendirilmesi


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier

# Target / Feature
target = "vehicle_type"
X = df.drop(columns=[target]).copy()
y = df[target].astype(int).copy()


X = X.drop(columns=["mean_passenger_by_route_hour"])


X["district"] = X["district"].fillna("UNKNOWN")

# Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Split shapes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

#  Columns
num_cols = ["hour", "passenger_count", "is_outlier", "is_peak_hour"]
cat_cols = ["route_code", "stop_code", "district"]

#  Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ]
)

#  Model
model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_depth=8,
    max_leaf_nodes=64,
    min_samples_leaf=40,
    early_stopping=True,
    random_state=42
)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", model)
])

#  Train
pipe.fit(X_train, y_train)

# Evaluate
pred = pipe.predict(X_test)

print("\n=== TEST Classification Report (LEAKAGE FIXED) ===")
print(classification_report(y_test, pred, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, pred))


Split shapes:
Train: (3219489, 7) Val: (689890, 7) Test: (689891, 7)

=== TEST Classification Report (LEAKAGE FIXED) ===
              precision    recall  f1-score   support

           1     1.0000    1.0000    1.0000    229963
           2     1.0000    1.0000    1.0000    229964
           3     1.0000    1.0000    1.0000    229964

    accuracy                         1.0000    689891
   macro avg     1.0000    1.0000    1.0000    689891
weighted avg     1.0000    1.0000    1.0000    689891


=== Confusion Matrix ===
[[229963      0      0]
 [     0 229964      0]
 [     0      0 229964]]


Deterministik özellikler çıkarılarak gerçekçi Gradient Boosting sınıflandırması


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier

#  Target / Feature
target = "vehicle_type"
X = df.drop(columns=[target]).copy()
y = df[target].astype(int).copy()

#  Deterministic leakage kolonları çıkar
X = X.drop(columns=["route_code", "stop_code", "mean_passenger_by_route_hour"])

# Missing fix
X["district"] = X["district"].fillna("UNKNOWN")

#  Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Split shapes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

#  Columns
num_cols = ["hour", "passenger_count", "is_outlier", "is_peak_hour"]
cat_cols = ["district"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ]
)

#  Model
model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_depth=6,
    max_leaf_nodes=31,
    min_samples_leaf=60,
    early_stopping=True,
    random_state=42
)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", model)
])

#  Train
pipe.fit(X_train, y_train)

#  Evaluate
pred = pipe.predict(X_test)

print("\n=== TEST Classification Report (REALISTIC) ===")
print(classification_report(y_test, pred, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, pred))


Split shapes:
Train: (3219489, 5) Val: (689890, 5) Test: (689891, 5)

=== TEST Classification Report (REALISTIC) ===
              precision    recall  f1-score   support

           1     0.7614    0.9079    0.8282    229963
           2     0.7563    0.6140    0.6778    229964
           3     0.7439    0.7408    0.7423    229964

    accuracy                         0.7542    689891
   macro avg     0.7539    0.7542    0.7494    689891
weighted avg     0.7539    0.7542    0.7494    689891


=== Confusion Matrix ===
[[208774  14687   6502]
 [ 36612 141202  52150]
 [ 28798  30811 170355]]


PyTorch modeli için veri hazırlama ve ön işleme adımları


In [5]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Target
target = "vehicle_type"

# Feature set 
X = df.drop(columns=[target, "route_code", "stop_code", "mean_passenger_by_route_hour"]).copy()
y = df[target].copy()

# Missing fix
X["district"] = X["district"].fillna("UNKNOWN")

# Encode district
le = LabelEncoder()
X["district"] = le.fit_transform(X["district"])

# Train / Val / Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Scale numeric
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Labels -> 0,1,2
y_train = y_train.values - 1
y_val   = y_val.values - 1
y_test  = y_test.values - 1

print("Shapes:")
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Shapes:
Train: (3219489, 5)
Val: (689890, 5)
Test: (689891, 5)


 PyTorch tabanlı derin öğrenme modeli için veri hazırlama ve DataLoader oluşturma


In [6]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

#  Feature set 
target = "vehicle_type"
X = df.drop(columns=[target, "route_code", "stop_code", "mean_passenger_by_route_hour"]).copy()
y = df[target].copy()

# district missing fix
X["district"] = X["district"].fillna("UNKNOWN")

# district encode
le = LabelEncoder()
X["district"] = le.fit_transform(X["district"].astype(str))


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


y_train = (y_train.values - 1).astype(np.int64)
y_val   = (y_val.values - 1).astype(np.int64)
y_test  = (y_test.values - 1).astype(np.int64)


class TabDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = TabDS(X_train, y_train)
val_ds   = TabDS(X_val, y_val)
test_ds  = TabDS(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8192, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=8192, shuffle=False)

print("Shapes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Torch device:", "cuda" if torch.cuda.is_available() else "cpu")
print("District classes:", len(le.classes_))


Shapes:
Train: (3219489, 5) Val: (689890, 5) Test: (689891, 5)
Torch device: cpu
District classes: 40


 Veri setinin yüklenmesi


In [7]:
import pandas as pd

df = pd.read_csv("birlesik_veri.csv")
print(df.shape)


(4599270, 9)


 PyTorch tabanlı derin öğrenme modeli için veri hazırlığı ve DataLoader oluşturulması


In [8]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

#  Feature set 
target = "vehicle_type"
X = df.drop(columns=[target, "route_code", "stop_code", "mean_passenger_by_route_hour"]).copy()
y = df[target].copy()

# district missing fix
X["district"] = X["district"].fillna("UNKNOWN")

# district encode
le = LabelEncoder()
X["district"] = le.fit_transform(X["district"].astype(str))

#  Train/Val/Test split 
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

#  Scale 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# labels: 1/2/3 -> 0/1/2
y_train = (y_train.values - 1).astype(np.int64)
y_val   = (y_val.values - 1).astype(np.int64)
y_test  = (y_test.values - 1).astype(np.int64)

#  Dataset/Dataloader 
class TabDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = TabDS(X_train, y_train)
val_ds   = TabDS(X_val, y_val)
test_ds  = TabDS(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8192, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=8192, shuffle=False)

print("Shapes:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Torch device:", "cuda" if torch.cuda.is_available() else "cpu")
print("District classes:", len(le.classes_))


Shapes:
Train: (3219489, 5) Val: (689890, 5) Test: (689891, 5)
Torch device: cpu
District classes: 40


Derin sinir ağı modeli eğitimi ve değerlendirmesi


In [10]:
import copy
import torch
from torch import nn
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"

#  Model (Deep NN + BatchNorm) 
class DeepNN(nn.Module):
    def __init__(self, in_dim=5, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.net(x)

model = DeepNN(in_dim=X_train.shape[1], n_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


def eval_loader(model, loader):
    model.eval()
    all_preds, all_true = [], []
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            bs = yb.size(0)
            total_loss += loss.item() * bs
            n += bs
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu().numpy())
            all_true.append(yb.cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_true  = np.concatenate(all_true)
    acc = accuracy_score(all_true, all_preds)
    f1m = f1_score(all_true, all_preds, average="macro")
    return total_loss / n, acc, f1m, all_true, all_preds

#  Train + Early Stopping 
max_epochs = 30
patience = 5
best_val_loss = float("inf")
best_state = None
pat = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    val_loss, val_acc, val_f1, _, _ = eval_loader(model, val_loader)
    print(f"Epoch {epoch:02d} | val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1_macro={val_f1:.4f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print("Early stopping!")
            break


model.load_state_dict(best_state)

# Test 
test_loss, test_acc, test_f1, y_true, y_pred = eval_loader(model, test_loader)
print("\n=== TEST RESULTS ===")
print(f"test_loss={test_loss:.4f} test_acc={test_acc:.4f} test_f1_macro={test_f1:.4f}")

print("\n=== Classification Report (0/1/2) ===")
print(classification_report(y_true, y_pred, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_true, y_pred))


Epoch 01 | val_loss=0.7600 val_acc=0.6724 val_f1_macro=0.6563
Epoch 02 | val_loss=0.7110 val_acc=0.6949 val_f1_macro=0.6829
Epoch 03 | val_loss=0.7123 val_acc=0.6927 val_f1_macro=0.6831
Epoch 04 | val_loss=0.6626 val_acc=0.7198 val_f1_macro=0.7137
Epoch 05 | val_loss=0.6540 val_acc=0.7122 val_f1_macro=0.7020
Epoch 06 | val_loss=0.6239 val_acc=0.7240 val_f1_macro=0.7144
Epoch 07 | val_loss=0.6106 val_acc=0.7288 val_f1_macro=0.7195
Epoch 08 | val_loss=0.6062 val_acc=0.7359 val_f1_macro=0.7286
Epoch 09 | val_loss=0.6009 val_acc=0.7415 val_f1_macro=0.7348
Epoch 10 | val_loss=0.6043 val_acc=0.7398 val_f1_macro=0.7335
Epoch 11 | val_loss=0.5952 val_acc=0.7426 val_f1_macro=0.7359
Epoch 12 | val_loss=0.5814 val_acc=0.7505 val_f1_macro=0.7447
Epoch 13 | val_loss=0.5899 val_acc=0.7492 val_f1_macro=0.7438
Epoch 14 | val_loss=0.5967 val_acc=0.7424 val_f1_macro=0.7351
Epoch 15 | val_loss=0.5697 val_acc=0.7513 val_f1_macro=0.7466
Epoch 16 | val_loss=0.5763 val_acc=0.7468 val_f1_macro=0.7382
Epoch 17

Sınıf bazlı performans metriklerini hesaplayan fonksiyon


In [11]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def per_class_metrics(y_true, y_pred):
    class_labels = ["Deniz", "Kara", "Raylı"]

    precision = precision_score(y_true, y_pred, average=None)
    recall    = recall_score(y_true, y_pred, average=None)
    f1        = f1_score(y_true, y_pred, average=None)
    
    # support = sınıfların test setindeki gerçek örnek sayısı
    support   = pd.Series(y_true).value_counts().sort_index().values

    df = pd.DataFrame({
        "Class": class_labels,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Support": support
    })

    return df


Sınıf bazlı metrik sonuç tablosu

In [12]:
df_deepnn_metrics = per_class_metrics(y_true, y_pred)
df_deepnn_metrics

,Class,Precision,Recall,F1-Score,Support
0,Deniz,0.757574,0.911660,0.827505,229963
1,Kara,0.754925,0.605686,0.672121,229964
2,Raylı,0.743794,0.739551,0.741666,229964
